# Construction PPE Detection — Train on Google Colab

Trains a YOLOv8 detector on the **PPE dataset** (`segp-fcn6m/ppe-yezzu-fwbjo`, ~33k train images, 5 classes: `Helmet`, `No-Helmet`, `No-Vest`, `Person`, `Vest`) and evaluates it on the held-out **test split (4,190 images)** with per-class precision / recall / F1 / mAP — flagging which classes clear **90%**.

**Why Colab:** the laptop ran out of *system RAM* on 33k images. Colab gives a stronger GPU (T4/L4/A100) and more RAM.

### Before you run
1. **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is free; L4/A100 on Pro are much faster).
2. Then **Runtime → Run all** (or run cells top to bottom).

> Free-tier T4 sessions can disconnect after a few hours. For a long run, either use Colab **Pro**, lower `EPOCHS`, or keep the **Google Drive** save on (Cell 7) so you can resume.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU — set Runtime > Change runtime type > GPU, then re-run.')

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics, roboflow
print('ultralytics', ultralytics.__version__, '| roboflow', roboflow.__version__)

## 3. Roboflow API key

**Recommended:** click the **🔑 key icon** in Colab's left sidebar, add a secret named `ROBOFLOW_API_KEY`, and enable notebook access. Otherwise paste it in the fallback line below (and rotate it afterward on Roboflow).

In [ ]:
import os
ROBOFLOW_API_KEY = ''
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')
assert ROBOFLOW_API_KEY, (
    'No Roboflow API key found. Add a Colab Secret named ROBOFLOW_API_KEY '
    '(key icon in the left sidebar, enable notebook access), or set the '
    'ROBOFLOW_API_KEY environment variable. Get your key at '
    'https://app.roboflow.com (Settings -> API).'
)

## 4. Download the dataset (~2.8 GB)

In [ ]:
import shutil, os
# Roboflow SKIPS the download if the target folder already exists (and would then
# print "Downloaded to: ..." in ~1s without fetching). Wipe it for a clean fetch.
shutil.rmtree('/content/ppe', ignore_errors=True)

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('segp-fcn6m').project('ppe-yezzu-fwbjo')
dataset = project.version(1).download('yolov8', location='/content/ppe')
DATA_DIR = dataset.location
print('Downloaded to:', DATA_DIR)
print('contains:', os.listdir(DATA_DIR))   # should list train, valid, test, data.yaml

## 5. Fix `data.yaml` paths

Roboflow's export uses relative paths (`../train/images`) that don't resolve. Rewrite with an absolute `path` and correct split sub-paths.

In [ ]:
import os, glob, yaml
# Roboflow may nest the export in a versioned subfolder, so LOCATE data.yaml
# instead of assuming it sits directly in DATA_DIR.
print('/content/ppe contains:', os.listdir('/content/ppe') if os.path.isdir('/content/ppe') else 'MISSING')
matches = glob.glob('/content/**/data.yaml', recursive=True)
assert matches, 'No data.yaml found — re-run the download cell (Cell 4) and confirm it finished.'
yml = matches[0]                    # the real path to data.yaml
DATA_DIR = os.path.dirname(yml)     # fix DATA_DIR to the actual dataset folder
spec = yaml.safe_load(open(yml))
spec['path'] = DATA_DIR
spec['train'] = 'train/images'
spec['val'] = 'valid/images'
spec['test'] = 'test/images'
yaml.safe_dump(spec, open(yml, 'w'), sort_keys=False)
print('data.yaml:', yml)
print('DATA_DIR :', DATA_DIR, '| classes:', spec['names'])
for k in ('train', 'val', 'test'):
    p = os.path.join(DATA_DIR, spec[k])
    print(f'  {k}: {p} -> exists: {os.path.isdir(p)}')

## 6. (Optional but recommended) Save runs to Google Drive

Persists checkpoints so a disconnected session can resume, and keeps `best.pt` after the runtime ends. Skip this cell to keep everything in the ephemeral `/content`.

In [ ]:
USE_DRIVE = True  # set False to skip Drive
PROJECT_DIR = '/content/runs'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/ppe_runs'
    os.makedirs(PROJECT_DIR, exist_ok=True)
print('runs ->', PROJECT_DIR)

## 7. Train

`batch=-1` lets ultralytics auto-pick the batch size for the detected GPU. On a free **T4**, ~40 epochs takes a few hours; lower `EPOCHS` or use Pro (L4/A100) to go faster. Early stopping (`patience`) ends it once the model plateaus. If the session dropped mid-run, set `RESUME = True` to continue from the last checkpoint.

In [ ]:
from ultralytics import YOLO

BASE_MODEL = 'yolov8s.pt'   # try 'yolov8m.pt' for more capacity (with this much data it can help)
EPOCHS = 40
IMGSZ = 640
RUN_NAME = 'ppe_s'
RESUME = False             # set True to resume an interrupted run of the same name

ckpt = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'last.pt')
model = YOLO(ckpt if (RESUME and os.path.isfile(ckpt)) else BASE_MODEL)

model.train(
    data=yml,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,              # auto batch size; or set a number e.g. 32
    patience=12,
    device=0,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    resume=RESUME,
    plots=True,
)
BEST = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt')
print('Best checkpoint:', BEST)

## 8. Evaluate on the held-out test split (4,190 images)

Authoritative per-class precision / recall / F1 / mAP@50 / mAP@50-95, with a 90% check per class.

In [ ]:
def evaluate(weights, tta=False):
    m = YOLO(weights)
    res = m.val(data=yml, split='test', imgsz=IMGSZ, augment=tta, device=0, verbose=False)
    box = res.box
    names = res.names
    idx = list(box.ap_class_index)
    p, r, f1 = list(box.p), list(box.r), list(box.f1)
    ap50, ap = list(box.ap50), list(box.ap)
    print('=' * 78)
    print(f" Per-class metrics on TEST{'  (TTA)' if tta else ''}")
    print('=' * 78)
    print(f"{'class':<12}{'precision':>11}{'recall':>10}{'F1':>9}{'mAP@50':>10}{'mAP50-95':>11}{'  90%?':>8}")
    n90 = 0
    for j, ci in enumerate(idx):
        name = names[int(ci)]
        row = dict(precision=float(p[j]), recall=float(r[j]), f1=float(f1[j]),
                   map50=float(ap50[j]), map5095=float(ap[j]))
        hit = all(row[k] >= 0.90 for k in ('precision', 'recall', 'f1', 'map50'))
        n90 += hit
        f = lambda x: f'{x*100:6.1f}%'
        print(f"{name:<12}{f(row['precision']):>11}{f(row['recall']):>10}{f(row['f1']):>9}"
              f"{f(row['map50']):>10}{f(row['map5095']):>11}{('  YES' if hit else ''):>8}")
    mp, mr = float(box.mp), float(box.mr)
    mf1 = 2*mp*mr/(mp+mr) if (mp+mr) else 0.0
    print('-' * 78)
    print(f"{'ALL (mean)':<12}{mp*100:10.1f}%{mr*100:9.1f}%{mf1*100:8.1f}%"
          f"{float(box.map50)*100:9.1f}%{float(box.map)*100:10.1f}%")
    print(f"\nClasses clearing 90% on P/R/F1/mAP@50: {n90}/{len(idx)}")
    return res

_ = evaluate(BEST)

## 9. (Optional) Test-time augmentation — usually +1-3 mAP for free

In [ ]:
_ = evaluate(BEST, tta=True)

## 10. Get the model back

If you used Drive (Cell 6), `best.pt` is already saved under `MyDrive/ppe_runs/ppe_s/weights/`. To also download it locally:

In [ ]:
from google.colab import files
files.download(BEST)
# Drop the downloaded best.pt into your local project as models/finetuned.pt (or models/ppe_s.pt)
# and evaluate with:  python eval_ppe.py --model models/ppe_s.pt --dataset-dir data/ppe_download